<a href="https://colab.research.google.com/github/pathilink/adyen_payment_optimization_case/blob/main/notebooks/04_authorization_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='#0ABF56'> Optimisation Data Analyst Case Study </font>

## <font color='#0ABF56'> 04 - Authorization Analysis </font>

# Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime
import seaborn as sns
from matplotlib import pyplot as plt

# Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/test/adyen/data/processed/adyen_transactions_analysis.csv')
df_ = df.copy()
df_.head()

,psp_reference,bin,scheme,issuername,shopper_interaction,avs_data_supplied,cvc_data_supplied,amount,raw_acquirer_response,creation_date,authorization,issuer_known,amount_range
0,1,400178.0,visa,BANCO DO BRASIL S.A.,Ecommerce,False,False,1.00,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 00:19:00,False,True,Until 4.16
1,2,486348.0,visa,FIRST ATLANTIC BANK LIMITED,Ecommerce,False,False,6.48,00 : Approved or completed successfully,2019-06-01 00:22:00,True,True,4.16 to 19.55
2,3,482481.0,visa,ITAU UNIBANCO S.A.,Ecommerce,False,True,4.00,06 : Error,2019-06-01 00:46:00,False,True,Until 4.16
3,4,439267.0,visa,CAIXA ECONOMICA FEDERAL,Ecommerce,False,True,2.76,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 01:02:00,False,True,Until 4.16
4,5,489347.0,visa,VTB BANK PJSC,Ecommerce,False,True,97.00,00 : Approved or completed successfully,2019-06-01 01:30:00,True,True,above 52.10


# Analysis

In [4]:
# filters all rejections from the database
df_refusals = df_[df_['authorization'] == False]

# global share
overall_ranking = (
    df_refusals['issuername']
    .value_counts()
    .reset_index()
)

overall_ranking.columns = ['issuername', 'total_refusals']

# the percentage that each bank accounts for of the total number of rejections
total_global_refusal = overall_ranking['total_refusals'].sum()
overall_ranking['share_refusals (%)'] = (overall_ranking['total_refusals'] / total_global_refusal * 100).round(2)

# overall_ranking.head(10)

In [5]:
# cumulative percentage
overall_ranking['cumulative_refusals (%)'] = overall_ranking['share_refusals (%)'].cumsum()

overall_ranking.head(11)

,issuername,total_refusals,share_refusals (%),cumulative_refusals (%)
0,ITAU UNIBANCO S.A.,31680,16.45,16.45
1,NU PAGAMENTOS SA,25294,13.14,29.59
2,BANCO SANTANDER (BRASIL) S.A.,24605,12.78,42.37
3,BANCO BRADESCO S.A.,22475,11.67,54.04
4,BANCO DO BRASIL S.A.,16818,8.73,62.77
5,CAIXA ECONOMICA FEDERAL,7706,4.00,66.77
6,BANCO ITAUCARD S.A.,7209,3.74,70.51
7,BANCO BRADESCARD S.A.,7065,3.67,74.18
8,BANCO SANTANDER S.A.,3959,2.06,76.24
9,HUB PAGAMENTOS S.A.,3267,1.70,77.94


Applying the Pareto Principle, I have ranked 11 institutions that account for almost 80% of declined transactions and generate losses.

1. High-Impact Group (“Big 5”)
    * Institutions: Itaú, Nubank, Santander, Bradesco and Banco do Brasil.
    * Impact: Together, they account for 62.77% of all rejections on the platform.
    * Conclusion: As they operate with massive volumes, small optimisations of 1% or 2% in the approval rate will generate more revenue than resolving 100% of the problems of all small banks combined.

2. Rule Alignment Block
    * Institutions: Caixa Econômica, Itaucard and Bradescard (Cumulative total: 74.18%).
    * Impact: They represent the second level of revenue loss.
    * Decision: They require a clinical eye on specific rules for co-branded cards (partnerships with retailers, such as Bradescard).

3. Final Funnel Block
    * Institutions: Hub Pagamentos, Banco Inter and Neon (accounting for 79.61% of the total).
    * Impact: Responsible for the final segment of the identified bottleneck.
    * Decision: Although some have poor individual approval rates (such as Banco Inter for transactions without a CVC), their overall impact is low.


In [6]:
top11 = overall_ranking.head(11)['issuername'].tolist()
top11

['ITAU UNIBANCO S.A.',
 'NU PAGAMENTOS SA',
 'BANCO SANTANDER (BRASIL) S.A.',
 'BANCO BRADESCO S.A.',
 'BANCO DO BRASIL S.A.',
 'CAIXA ECONOMICA FEDERAL',
 'BANCO ITAUCARD S.A.',
 'BANCO BRADESCARD S.A.',
 'BANCO SANTANDER S.A.',
 'HUB PAGAMENTOS S.A.',
 'BANCO INTER S.A.']

In [7]:
# total transaction volume (approved + declined)
total_transaction = len(df_)

# group by issuer
share_transactions = (
    df_['issuername']
    .value_counts()
    .reset_index()
)
share_transactions.columns = ['issuername', 'total_transactions']

# calculate share
share_transactions['share_transactions (%)'] = (share_transactions['total_transactions'] / total_transaction * 100).round(2)

# cumulative sum
share_transactions['cumulative_transactions (%)'] = share_transactions['share_transactions (%)'].cumsum()

ranking_11_transaction = share_transactions.query('issuername in @top11')

# ranking_11_transaction

In [8]:
df_authorization_analysis = pd.merge(
    ranking_11_transaction,
    overall_ranking.head(11),
    on='issuername',
    how='inner'
)

# df_authorization_analysis

* Itaú and Nubank: ensure that no technical changes disrupt their transaction flow, as they account for almost 40% of all transactions.

* Bradesco: has high transaction volume (almost 9%) and poor efficiency (11.67% decline rate). It is a good issuer on which to focus immediate efforts to recover transactions.

* Bradescard, Inter and Hub: these issuers’ systems are blocking their customers disproportionately. It is worth investigating the reasons that trigger the anti-fraud measures of these digital banks and store cards.

## raw_acquirer_response

In [9]:
# total number of transactions == False for the issuer with the specific raw_acquirer_response
grouped = (
    df_
    .query('authorization == False and issuername in @top11')
    .groupby(['issuername', 'raw_acquirer_response'])
    .size()
    .reset_index(name='declined_transactions')
)

# get total transactions per issuer from df_ (authorization True or False)
total_transactions_per_issuer = df_.groupby('issuername').size().reset_index(name='total_issuer_transactions')

# merge to get the total number of transactions for each issuer
grouped = pd.merge(grouped, total_transactions_per_issuer, on='issuername', how='left')

# calculate the rate
grouped['declined_transactions_rate'] = grouped['declined_transactions'] / grouped['total_issuer_transactions']

# applies the minimum volume filter and sorts the results
final_result = (
    grouped
    # .query('transactions >= 1000') # at least
    .sort_values(['issuername', 'declined_transactions_rate'], ascending=[True, False])
)

# final_result

In [10]:
most_frequent_by_issuer_refusal = final_result.groupby('issuername').first().reset_index()

most_frequent_by_issuer_refusal = most_frequent_by_issuer_refusal.rename(columns={
    'raw_acquirer_response': 'most_common_refusal',
    'declined_transactions': 'total_most_common_refusal',
    'declined_transactions_rate': 'most_common_refusal (%)'
})

most_frequent_by_issuer_refusal['most_common_refusal (%)'] = (most_frequent_by_issuer_refusal['most_common_refusal (%)'] * 100).round(2)

most_frequent_by_issuer_refusal[['issuername', 'most_common_refusal', 'total_most_common_refusal', 'most_common_refusal (%)']]


,issuername,most_common_refusal,total_most_common_refusal,most_common_refusal (%)
0,BANCO BRADESCARD S.A.,05 : Do not honor,5050,28.92
1,BANCO BRADESCO S.A.,05 : Do not honor,11018,13.33
2,BANCO DO BRASIL S.A.,57 : Transaction not permitted to issuer/cardh...,5981,7.88
3,BANCO INTER S.A.,05 : Do not honor,2215,28.94
4,BANCO ITAUCARD S.A.,57 : Transaction not permitted to issuer/cardh...,2302,4.03
5,BANCO SANTANDER (BRASIL) S.A.,51 : Insufficient funds/over credit limit,9695,7.87
6,BANCO SANTANDER S.A.,05 : Do not honor,950,3.06
7,CAIXA ECONOMICA FEDERAL,05 : Do not honor,4036,16.28
8,HUB PAGAMENTOS S.A.,51 : Insufficient funds/over credit limit,2789,40.91
9,ITAU UNIBANCO S.A.,62 : Restricted card,18253,9.13


In [11]:
df_authorization_analysis = pd.merge(
    ranking_11_transaction,
    overall_ranking.head(11),
    on='issuername',
    how='inner'
)

# Drop existing refusal-related columns from df_authorization_analysis to avoid merge conflicts
# This step is now primarily for robustness, as the DataFrame is re-initialized
df_authorization_analysis = df_authorization_analysis.drop(columns=[
    'most_common_refusal',
    'declined_transactions',
    'total_issuer_transactions',
    'most_common_refusal (%)',
    'total_most_common_refusal'
], errors='ignore')

df_authorization_analysis = pd.merge(
    df_authorization_analysis,
    most_frequent_by_issuer_refusal,
    on='issuername',
    how='inner'
)

df_authorization_analysis = df_authorization_analysis[[
    'issuername', 'total_transactions', 'share_transactions (%)',
    'cumulative_transactions (%)', 'total_refusals', 'share_refusals (%)',
    'cumulative_refusals (%)',
    'most_common_refusal', 'total_most_common_refusal', 'most_common_refusal (%)'
]]

# df_authorization_analysis

**The main rejections among the top 11 issuers**

| Reason | Description |
|:-|:-|
| 05 : Do not honor| 	The card issuer requests to retain the card. This can be due to a suspected counterfeit or stolen card. <br>This reason is used in an ecommerce environment although it originates from an in-person payments environment. |
| 51 : Insufficient funds/over credit limit | Insufficient funds in the cardholder's account. <br>The shopper can try again after adding funds to their bank account, or use another payment method. |
| 57 : Transaction not permitted to issuer/cardholder | Decline |
| 62 : Restricted card | 	The card issuer has restricted where the card can be used. For example, because of embargoes. |

## shopper_interaction

In [12]:
# approved + declined
(
    df_
    .query('issuername in @top11')
    .groupby(
      ['issuername', 'shopper_interaction']
      )['authorization'].mean()
    .reset_index()
)

,issuername,shopper_interaction,authorization
0,BANCO BRADESCARD S.A.,ContAuth,0.623634
1,BANCO BRADESCARD S.A.,Ecommerce,0.585922
2,BANCO BRADESCO S.A.,ContAuth,0.880288
3,BANCO BRADESCO S.A.,Ecommerce,0.579787
4,BANCO DO BRASIL S.A.,ContAuth,0.871296
5,BANCO DO BRASIL S.A.,Ecommerce,0.732501
6,BANCO INTER S.A.,ContAuth,0.745890
7,BANCO INTER S.A.,Ecommerce,0.540039
8,BANCO ITAUCARD S.A.,ContAuth,0.912856
9,BANCO ITAUCARD S.A.,Ecommerce,0.820809


The main strategy here would be to encourage customers to save their card details on the website or app (becoming a ContAuth user). Doing so will increase the likelihood of approval, particularly for customers with Bradesco and Inter cards, where the difference in approval rates between the two methods can exceed 20 percentage points.

In [13]:
# only declined
df_shopper_interaction_refusal = (
    df_.query("issuername in @top11")
    # the ~ operator converts True (pass) to False (0) and False (fail) to True (1)
    .assign(recusa=lambda x: ~x["authorization"])
    .groupby(["issuername", "shopper_interaction"])["recusa"]
    .mean()
    # turn ('ContAuth', 'Ecommerce') into columns
    .unstack(level="shopper_interaction")
    .multiply(100)
    .round(2)
    .reset_index()
    .rename(columns={"ContAuth": "ContAuth_refusal (%)", "Ecommerce": "Ecommerce_refusal (%)"})
)

df_shopper_interaction_refusal

shopper_interaction,issuername,ContAuth_refusal (%),Ecommerce_refusal (%)
0,BANCO BRADESCARD S.A.,37.64,41.41
1,BANCO BRADESCO S.A.,11.97,42.02
2,BANCO DO BRASIL S.A.,12.87,26.75
3,BANCO INTER S.A.,25.41,46.00
4,BANCO ITAUCARD S.A.,8.71,17.92
5,BANCO SANTANDER (BRASIL) S.A.,11.23,27.39
6,BANCO SANTANDER S.A.,7.25,19.70
7,CAIXA ECONOMICA FEDERAL,21.56,36.78
8,HUB PAGAMENTOS S.A.,45.12,49.08
9,ITAU UNIBANCO S.A.,9.18,22.87


In [14]:
df_authorization_analysis = pd.merge(
    df_authorization_analysis,
    df_shopper_interaction_refusal,
    on='issuername',
    how='inner'
)

# df_authorization_analysis

## cvc_data_supplied

In [15]:
(
    df_
    .query('issuername in @top11')
    .groupby(
      ['issuername', 'cvc_data_supplied']
      )['authorization'].mean()
    .reset_index()
)

,issuername,cvc_data_supplied,authorization
0,BANCO BRADESCARD S.A.,False,0.603392
1,BANCO BRADESCARD S.A.,True,0.587101
2,BANCO BRADESCO S.A.,False,0.781750
3,BANCO BRADESCO S.A.,True,0.569006
4,BANCO DO BRASIL S.A.,False,0.842051
5,BANCO DO BRASIL S.A.,True,0.700820
6,BANCO INTER S.A.,False,0.733351
7,BANCO INTER S.A.,True,0.431026
8,BANCO ITAUCARD S.A.,False,0.880158
9,BANCO ITAUCARD S.A.,True,0.846511


At the vast majority of banks, transactions WITHOUT a CVC (cvc_data_supplied == False) have higher approval rates than transactions WITH a CVC (True).

The only exception is Nubank, whose risk model may place greater value on physical presence or the provision of complete details at the time of purchase.

In [16]:
# only declined
df_cvc_refusal = (
    df_.query("issuername in @top11")
    # the ~ operator converts True (pass) to False (0) and False (fail) to True (1)
    .assign(recusa=lambda x: ~x["authorization"])
    .groupby(["issuername", "cvc_data_supplied"])["recusa"]
    .mean()
    # turn ('with cvc', 'without cvc') into columns
    .unstack(level="cvc_data_supplied")
    .multiply(100)
    .round(2)
    .reset_index()
    .rename(columns={False: "without_cvc_refusal (%)", True: "with_cvc_refusal (%)"})
)

df_cvc_refusal

cvc_data_supplied,issuername,without_cvc_refusal (%),with_cvc_refusal (%)
0,BANCO BRADESCARD S.A.,39.66,41.29
1,BANCO BRADESCO S.A.,21.82,43.10
2,BANCO DO BRASIL S.A.,15.79,29.92
3,BANCO INTER S.A.,26.66,56.90
4,BANCO ITAUCARD S.A.,11.98,15.35
5,BANCO SANTANDER (BRASIL) S.A.,15.10,31.82
6,BANCO SANTANDER S.A.,10.04,22.84
7,CAIXA ECONOMICA FEDERAL,25.57,39.85
8,HUB PAGAMENTOS S.A.,47.11,49.05
9,ITAU UNIBANCO S.A.,12.44,25.76


In [17]:
df_authorization_analysis = pd.merge(
    df_authorization_analysis,
    df_cvc_refusal,
    on='issuername',
    how='inner'
)

# df_authorization_analysis

## amount

In [18]:
(
    df_
    .query('issuername in @top11')
    .groupby(
    ['issuername', 'amount_range'], observed=True
    )['authorization'].mean()
    .reset_index()
)

,issuername,amount_range,authorization
0,BANCO BRADESCARD S.A.,19.55 to 52.10,0.617506
1,BANCO BRADESCARD S.A.,4.16 to 19.55,0.598658
2,BANCO BRADESCARD S.A.,Until 4.16,0.611296
3,BANCO BRADESCARD S.A.,above 52.10,0.541468
4,BANCO BRADESCO S.A.,19.55 to 52.10,0.796904
5,BANCO BRADESCO S.A.,4.16 to 19.55,0.731760
6,BANCO BRADESCO S.A.,Until 4.16,0.579778
7,BANCO BRADESCO S.A.,above 52.10,0.772840
8,BANCO DO BRASIL S.A.,19.55 to 52.10,0.858394
9,BANCO DO BRASIL S.A.,4.16 to 19.55,0.792666


In [19]:
df_amount_range = (
    df_.query("issuername in @top11")
    # the ~ operator converts True (pass) to False (0) and False (fail) to True (1)
    .assign(recusa=lambda x: ~x["authorization"])
    .groupby(["issuername", "amount_range"])["recusa"]
    .mean()
    # turn ('with cvc', 'without cvc') into columns
    .unstack(level="amount_range")
    .multiply(100)
    .round(2)
    .reset_index()
)

df_amount_range = df_amount_range.rename(columns={
    'Until 4.16': 'Until 4.16_refusal (%)',
    '4.16 to 19.55': '4.16 to 19.55_refusal (%)',
    '19.55 to 52.10': '19.55 to 52.10_refusal (%)',
    'above 52.10': 'above 52.10_refusal (%)'
})

# removes the residual column names from the index to clean up the DataFrame’s structure
df_amount_range.columns.name = None

# reorder columns
df_amount_range = df_amount_range[
    [
        "issuername",
        "Until 4.16_refusal (%)",
        "4.16 to 19.55_refusal (%)",
        "19.55 to 52.10_refusal (%)",
        "above 52.10_refusal (%)",
    ]
]

df_amount_range

,issuername,Until 4.16_refusal (%),4.16 to 19.55_refusal (%),19.55 to 52.10_refusal (%),above 52.10_refusal (%)
0,BANCO BRADESCARD S.A.,38.87,40.13,38.25,45.85
1,BANCO BRADESCO S.A.,42.02,26.82,20.31,22.72
2,BANCO DO BRASIL S.A.,33.20,20.73,14.16,21.22
3,BANCO INTER S.A.,53.47,38.20,23.17,29.60
4,BANCO ITAUCARD S.A.,14.86,11.96,10.26,14.03
5,BANCO SANTANDER (BRASIL) S.A.,28.39,18.75,13.08,18.91
6,BANCO SANTANDER S.A.,19.93,10.15,9.24,13.31
7,CAIXA ECONOMICA FEDERAL,44.33,30.03,21.25,27.37
8,HUB PAGAMENTOS S.A.,45.82,50.62,45.64,53.55
9,ITAU UNIBANCO S.A.,23.67,15.25,10.61,14.89


The highest proportion of rejections is found among the lowest amounts (up to R$4.16).

Transactions involving very small amounts trigger alerts in the banks’ anti-fraud systems. Scammers often make ‘test purchases’ (transactions involving cents or a few reais) to check whether a card is active before carrying out a larger scam. Knowing this, banks block transactions in this range.

In [20]:
df_authorization_analysis = pd.merge(
    df_authorization_analysis,
    df_amount_range,
    on='issuername',
    how='inner'
)

df_authorization_analysis

,issuername,total_transactions,share_transactions (%),cumulative_transactions (%),total_refusals,share_refusals (%),cumulative_refusals (%),most_common_refusal,total_most_common_refusal,most_common_refusal (%),ContAuth_refusal (%),Ecommerce_refusal (%),without_cvc_refusal (%),with_cvc_refusal (%),Until 4.16_refusal (%),4.16 to 19.55_refusal (%),19.55 to 52.10_refusal (%),above 52.10_refusal (%)
0,ITAU UNIBANCO S.A.,199821,21.41,21.41,31680,16.45,16.45,62 : Restricted card,18253,9.13,9.18,22.87,12.44,25.76,23.67,15.25,10.61,14.89
1,NU PAGAMENTOS SA,159244,17.06,38.47,25294,13.14,29.59,05 : Do not honor,16660,10.46,12.93,18.46,16.81,12.99,15.28,16.90,14.74,17.03
2,BANCO SANTANDER (BRASIL) S.A.,123232,13.20,51.67,24605,12.78,42.37,51 : Insufficient funds/over credit limit,9695,7.87,11.23,27.39,15.10,31.82,28.39,18.75,13.08,18.91
3,BANCO BRADESCO S.A.,82630,8.85,60.52,22475,11.67,54.04,05 : Do not honor,11018,13.33,11.97,42.02,21.82,43.10,42.02,26.82,20.31,22.72
4,BANCO DO BRASIL S.A.,75933,8.14,68.66,16818,8.73,62.77,57 : Transaction not permitted to issuer/cardh...,5981,7.88,12.87,26.75,15.79,29.92,33.20,20.73,14.16,21.22
5,BANCO ITAUCARD S.A.,57072,6.11,74.77,7209,3.74,70.51,57 : Transaction not permitted to issuer/cardh...,2302,4.03,8.71,17.92,11.98,15.35,14.86,11.96,10.26,14.03
6,BANCO SANTANDER S.A.,31029,3.32,78.09,3959,2.06,76.24,05 : Do not honor,950,3.06,7.25,19.70,10.04,22.84,19.93,10.15,9.24,13.31
7,CAIXA ECONOMICA FEDERAL,24795,2.66,80.75,7706,4.00,66.77,05 : Do not honor,4036,16.28,21.56,36.78,25.57,39.85,44.33,30.03,21.25,27.37
8,BANCO BRADESCARD S.A.,17462,1.87,84.52,7065,3.67,74.18,05 : Do not honor,5050,28.92,37.64,41.41,39.66,41.29,38.87,40.13,38.25,45.85
9,BANCO INTER S.A.,7654,0.82,87.77,3220,1.67,79.61,05 : Do not honor,2215,28.94,25.41,46.00,26.66,56.90,53.47,38.20,23.17,29.60


# Conclusion

**Where is there the greatest potential for uplift?**

The biggest opportunity lies in improving the security of the e-commerce payment process.

* Smart CVC: configure the website to require the CVC when the card is from Nubank, as their system tends to reject purchases that do not include this information.

* Encourage customers to save their card details on the e-commerce site: saved cards are viewed as secure by banks and are more likely to be approved.

* End of test charges: banks such as Inter believe that testing whether a customer’s card works with a small amount is indicative of fraudulent behaviour and block the purchase.

# Download

In [21]:
output_path = '/content/drive/MyDrive/test/adyen/data/processed/adyen_transactions_analysis_summary.csv'

df_authorization_analysis.to_csv(output_path, index=False)